In [ ]:
!pip install lightgbm -q

In [ ]:
!pip install seaborn -q

In [ ]:
# ==================================================
# 0. IMPORTS E CONFIGURAÇÕES
# ==================================================
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import json
import os
import ocifs
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import gc

# Configuração de semente aleatória (Essencial para reprodutibilidade)
SEED = 42

# Configuração de Caminhos
BUCKET = "cs-00"
NAMESPACE = "griphcnm8tdc"
PATH_MODELS = f"oci://{BUCKET}@{NAMESPACE}/models/v_06_final"

print(f"Caminho de Saída: {PATH_MODELS}")

In [ ]:
# ==================================================
# 1. LEITURA INTELIGENTE (Só o que precisamos)
# ==================================================
# ==================================================
# 1. DEFINIÇÃO DAS VARIÁVEIS
# ==================================================
# IDs e Metadados
ids = ['NUM_CPF', 'SAFRA', 'GRUPO_ANALISE']

# Features Numéricas (Scores)
features_scores = ['SCORE_01', 'SCORE_02']

# Features Categóricas (Score) - flag_mig2
features_cat_score = ['flag_mig2']

# Target
target = 'FPD'

# Features Telco + Cadastro (Automático - busca nomes sem carregar dados)
import pyarrow.parquet as pq

fs = ocifs.OCIFileSystem()
# Lê apenas o schema (estrutura), sem carregar dados na memória
schema = pq.read_schema("oci://cs-00@griphcnm8tdc/raw/processed/abt_master.parquet", filesystem=fs)
all_cols = schema.names
features_telco = [c for c in all_cols if c.startswith('tel_')]
features_cad = [c for c in all_cols if c.startswith('cad_')]
features_recarga = [c for c in all_cols if c.startswith('rec_')]
features_atr = [c for c in all_cols if c.startswith('atr_')]
features_pag = [c for c in all_cols if c.startswith('pag_')]

# Atualizar a lista de carregamento incluindo as novas
cols_to_load = list(set(ids + features_scores + features_telco + features_cad + 
                        features_recarga + features_atr + features_pag + 
                        features_cat_score + [target]))

print(f"Carregando {len(cols_to_load)} colunas de {len(all_cols)} disponíveis...")
df = pd.read_parquet("oci://cs-00@griphcnm8tdc/raw/processed/abt_master.parquet", columns=cols_to_load)

In [ ]:
# Colunas que estão no arquivo mas não foram carregadas
missing_cols = [c for c in all_cols if c not in cols_to_load]
print(f"Colunas NÃO carregadas: {missing_cols}")

In [ ]:
# ==================================================
# 2. TRATAMENTO UNIFICADO E CORRIGIDO (OTIMIZADO)
# ==================================================
import gc
import numpy as np
import pandas as pd

print("Iniciando tratamento otimizado...")

# --- 1. TELCO ---
cols_telco = [c for c in df.columns if c.startswith('tel_')]
# Otimização: Converte direto, sem apply
df[cols_telco] = df[cols_telco].astype(float) # Tenta converter rápido

# Cria flags e limpa (mantendo sua lógica)
for col in cols_telco:
    df[f'{col}_is_304'] = (df[col] == 304).astype('int8')
    df[f'{col}_is_303'] = (df[col] == 303).astype('int8')
    df[f'{col}_is_zero'] = (df[col] == 0).astype('int8')
    df[f'{col}_is_100'] = (df[col] == 100).astype('int8')
    
    # Limpa e converte para float32
    df[col] = df[col].replace([304, 303, 0, 100], np.nan).astype('float32')

# --- 2. CADASTRO ---
cols_constantes = ['cad_var_06', 'cad_var_18', 'cad_var_19', 'cad_var_20', 
                   'cad_var_21', 'cad_var_22', 'cad_var_23']
df = df.drop(columns=[c for c in cols_constantes if c in df.columns], errors='ignore')

# Datas Cadastro
if 'cad_DATADENASCIMENTO' in df.columns:
    df['cad_DATADENASCIMENTO'] = pd.to_datetime(df['cad_DATADENASCIMENTO'], format='%d/%m/%Y', errors='coerce')
    df['cad_IDADE_DIAS'] = (pd.to_datetime('2025-01-01') - df['cad_DATADENASCIMENTO']).dt.days
    df = df.drop(columns=['cad_DATADENASCIMENTO'])

cols_cat_fix = ['cad_CEP_3_digitos', 'cad_STATUSRF', 'cad_var_02', 'cad_var_10', 'cad_var_15', 'cad_var_24', 'cad_var_25']
for col in cols_cat_fix:
    if col in df.columns: df[col] = df[col].astype('category')

# Numéricos Cadastro
cols_num_cad = ['cad_var_03', 'cad_var_04', 'cad_var_05', 'cad_var_07', 'cad_var_08', 
                'cad_var_09', 'cad_var_11', 'cad_var_14', 'cad_var_16', 'cad_var_17']
for col in cols_num_cad:
    if col in df.columns: 
        # Usa errors='coerce' para segurança
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

# Datas Cadastro (var_12/13)
for col in ['cad_var_12', 'cad_var_13']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
        df[f'{col}_dias'] = (pd.to_datetime('2025-01-01') - df[col]).dt.days
        df = df.drop(columns=[col])

# --- 3. RECARGA E ATRASO/PAGAMENTO (VERSÃO TURBO) ---
cols_rec = [c for c in df.columns if c.startswith('rec_')]
cols_atr = [c for c in df.columns if c.startswith('atr_')]
cols_pag = [c for c in df.columns if c.startswith('pag_')]
all_other_cols = cols_rec + cols_atr + cols_pag

# Estratégia: Separar numéricos de objetos para não rodar to_datetime em números desnecessariamente

# 3.1 Tratar colunas já numéricas (performance máxima)
# Seleciona apenas as que existem e são numéricas
existing_num_cols = [c for c in all_other_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

for col in existing_num_cols:
    # Flag de zero
    df[f'{col}_is_zero'] = (df[col] == 0).astype('int8')
    # Reduz memória
    if df[col].dtype == 'float64': 
        df[col] = df[col].astype('float32')
    elif df[col].dtype == 'int64': 
        df[col] = pd.to_numeric(df[col], downcast='integer') # Reduz int64 para int32/16/8 se possível

# 3.2 Tratar colunas objeto (string)
# Seleciona apenas as que existem e são objeto/texto
existing_obj_cols = [c for c in all_other_cols if c in df.columns and df[c].dtype == 'object']

print(f"Convertendo {len(existing_obj_cols)} colunas de texto (possíveis datas)...")

for col in existing_obj_cols:
    # Tenta converter para data
    # Otimização: Se a coluna tiver muitos valores únicos e parecer data, converte.
    temp_date = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
    
    # Se a conversão for bem sucedida na maioria dos casos (menos de 50% NaT), assumimos que é data
    if temp_date.notna().sum() / len(df) > 0.5:
        df[f'{col}_dias'] = (pd.to_datetime('2025-01-01') - temp_date).dt.days
        df = df.drop(columns=[col])
    else:
        # Se não for data, converte para Category (o que resolve seu erro do LightGBM!)
        df[col] = df[col].astype('category')

# Libera memória
gc.collect()
print("Tratamento concluído.")

In [ ]:
df.shape

In [ ]:
# ==================================================
# SALVAMENTO DA ABT tratada
# ==================================================
print("\nSalvando ABT tratada...")

# Caminho no OCI (crie uma pasta processed se não existir)
path_abt = "oci://cs-00@griphcnm8tdc/processed/abt_tratada.parquet"

df.to_parquet(path_abt, index=False)

print(f"✅ Base salva com sucesso em: {path_abt}")oci://cs-00@griphcnm8tdc/processed/abt_tratada.parquet

In [ ]:
df = pd.read_parquet("oci://cs-00@griphcnm8tdc/processed/abt_tratada.parquet"

In [ ]:
# ==================================================
# 3. SPLIT E LIBERAÇÃO DE MEMÓRIA CRITICAL
# ==================================================
# Filtra público
df['SAFRA'] = df['SAFRA'].astype(str)
train = df[df['SAFRA'].isin(['202410', '202411', '202412'])].dropna(subset=[target])
oos = df[df['SAFRA'] == '202501'].dropna(subset=[target])

print(f"Volume Treino: {train.shape[0]}")
print(f"Volume OOS: {oos.shape[0]}")

# A MÁGICA: Apagar o DF original da memória
del df
gc.collect() # Força a limpeza da lixeira do Python

print("Memória liberada. Pronto para treinar.")

In [ ]:
# ==================================================
# 4. PREPARAÇÃO DAS FEATURES (INCREMENTO 05 - + Recarga)
# ==================================================

# 1. Definição das listas de features
features_scores = ['SCORE_01', 'SCORE_02']
features_cat_score = ['flag_mig2']

# 2. Features Telco dinamicamente
features_telco_orig = [c for c in train.columns if c.startswith('tel_') and '_is_' not in c]
features_telco_flags = [c for c in train.columns if c.startswith('tel_') and '_is_' in c]

# 3. Features Cadastro (Novo Incremento)
# Categóricas (Lista fixa baseada no tratamento)
features_cat_cad = ['cad_var_02', 'cad_var_10', 'cad_var_15', 'cad_var_24', 'cad_var_25']
features_cat_cad = [c for c in features_cat_cad if c in train.columns] # Garante que existe

# Numéricas (Pega todas as 'cad_' que sobraram, incluindo as _dias criadas)
# Exclui as categóricas da lista numérica
features_cad_num = [c for c in train.columns if c.startswith('cad_') and c not in features_cat_cad]

# 4. Features Recarga
features_rec_num = [c for c in train.columns if c.startswith('rec_') and train[c].dtype.name != 'category']
features_rec_flags = [c for c in train.columns if c.startswith('rec_') and '_is_' in c]
features_rec_cat = [c for c in train.columns if c.startswith('rec_') and train[c].dtype.name == 'category']

# Features Atraso e Pagamento
features_atr_pag_num = [c for c in train.columns if (c.startswith('atr_') or c.startswith('pag_')) and train[c].dtype.name != 'category']
features_atr_pag_flags = [c for c in train.columns if (c.startswith('atr_') or c.startswith('pag_')) and '_is_' in c]
features_atr_pag_cat = [c for c in train.columns if (c.startswith('atr_') or c.startswith('pag_')) and train[c].dtype.name == 'category']


# 5. Garantir tipo Categorical (Treino e OOS)
# Unificamos todas as categóricas (Score + Cadastro)
all_cat_features = features_cat_score + features_cat_cad + features_rec_cat + features_atr_pag_cat

for col in all_cat_features:
    if col in train.columns:
        train[col] = train[col].astype('category')
    if col in oos.columns:
        oos[col] = oos[col].astype('category')

# 5. Consolidar lista final de features
features_v06 = features_scores + features_telco_orig + features_telco_flags + features_cad_num + all_cat_features + features_rec_num + features_rec_flags + features_rec_cat + features_atr_pag_num + features_atr_pag_flags + features_atr_pag_cat
features_v06 = list(set(features_v06)) # Remove duplicatas

# Segurança: Remover target e chaves
features_v06 = [f for f in features_v06 if f not in ['FPD', 'NUM_CPF', 'SAFRA', 'GRUPO_ANALISE']]

print(f"Total de features selecionadas: {len(features_v06)}")

# 6. Separar X e y
X_train = train[features_v06]
y_train = train[target]

X_oos = oos[features_v06]
y_oos = oos[target]


In [ ]:
# ==================================================
# CHECK PRÉ-TREINO
# ==================================================
print(f"Shape Treino: {X_train.shape} | Target Mean: {y_train.mean():.2%}")
print(f"Shape OOS:    {X_oos.shape} | Target Mean: {y_oos.mean():.2%}")

# Verifica se há colunas de texto (Object) que impediriam o LightGBM
cols_obj = X_train.select_dtypes(include='object').columns.tolist()
if cols_obj:
    print(f"⚠️ ALERTA: Colunas de texto (Object) detectadas: {cols_obj}")
else:
    print("✅ Tipos de dados OK (sem colunas Object).")

# Contagem de Nulos (LightGBM lida com isso, mas é bom saber)
print(f"Total Nulos no Treino: {X_train.isnull().sum().sum()}")

In [ ]:
# Identifica e remove colunas de data restantes
cols_data = X_train.select_dtypes(include=['datetime64']).columns
X_train = X_train.drop(columns=cols_data)
X_oos = X_oos.drop(columns=cols_data)

In [ ]:
print(f"Shape Treino: {X_train.shape} | Target Mean: {y_train.mean():.2%}")
print(f"Shape OOS:    {X_oos.shape} | Target Mean: {y_oos.mean():.2%}")


In [ ]:
# ==================================================
# 5. MODELAGEM: LIGHTGBM (CORRIGIDO E SEGURO)
# ==================================================
import gc

# 1. Extrair o Target (y) - Faça isso ANTES de modificar o dataframe
y_train = train[target]
y_oos = oos[target]

# 2. OTIMIZAÇÃO DE MEMÓRIA SEGURA (Idempotente)
# Em vez de dropar o que "não é feature", vamos manter APENAS o que é feature + target
# Se rodar de novo, ele apenas repete a seleção, não quebra.

cols_to_keep = features_v06 + [target] # Mantém features e o target temporariamente
train = train[cols_to_keep]
oos = oos[cols_to_keep]

# Agora removemos o target para ficar apenas com X
train = train.drop(columns=[target])
oos = oos.drop(columns=[target])

# X agora é igual a train
X_train = train
X_oos = oos

# 3. Reduzir precisão numérica (float64 -> float32)
for col in X_train.select_dtypes(include=['float64', 'float32']).columns:
    X_train[col] = X_train[col].astype('float32')
    if col in X_oos.columns:
        X_oos[col] = X_oos[col].astype('float32')

# 4. Garantir Categóricas (Sem cat.codes, apenas category)
# (Isso substitui o passo 4 antigo que estava causando problema)
for col in all_cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')
    if col in X_oos.columns:
        X_oos[col] = X_oos[col].astype('category')

# 5. Configurar Modelo
model = lgb.LGBMClassifier(
    random_state=SEED,
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8, # Aumentado conforme sugestão anterior, depois testar 8
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    objective='binary',
    metric='auc',
    n_jobs=4,
    verbose=-1,
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train) # Adicionado conforme sugestão
)

# --- CORREÇÃO DE EMERGÊNCIA: REMOVER DATAS ANTES DE TREINAR ---

# 1. Identificar quem são as colunas de data no Treino e OOS
dt_cols_train = X_train.select_dtypes(include=['datetime64']).columns.tolist()
dt_cols_oos = X_oos.select_dtypes(include=['datetime64']).columns.tolist()

print(f"⚠️ Encontradas colunas de data no Treino: {dt_cols_train}")
print(f"⚠️ Encontradas colunas de data na OOS: {dt_cols_oos}")

# 2. Remover forçadamente
if dt_cols_train:
    X_train = X_train.drop(columns=dt_cols_train)
if dt_cols_oos:
    X_oos = X_oos.drop(columns=dt_cols_oos)

print("Colunas de data removidas. Prosseguindo para o fit...")

print("Iniciando treinamento...")
model.fit(
    X_train, y_train,
    eval_set=[(X_oos, y_oos)],
    eval_metric='auc',
    categorical_feature=all_cat_features, # Adicionado conforme sugestão
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
)

In [ ]:
# ==================================================
# 6. MÉTRICAS E AVALIAÇÃO
# ==================================================
def calculate_ks(y_true, y_proba):
    """Calcula o Kolmogorov-Smirnov"""
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    ks = max(tpr - fpr)
    return ks

# Previsões
y_pred_train = model.predict_proba(X_train)[:, 1]
y_pred_oos = model.predict_proba(X_oos)[:, 1]

# Cálculos
auc_train = roc_auc_score(y_train, y_pred_train)
auc_oos = roc_auc_score(y_oos, y_pred_oos)

ks_train = calculate_ks(y_train, y_pred_train)
ks_oos = calculate_ks(y_oos, y_pred_oos)

gini_train = 2 * auc_train - 1
gini_oos = 2 * auc_oos - 1

# Impressão dos Resultados
print("\n" + "="*30)
print("   RESULTADOS DO MODELO V_06")
print("="*30)
print(f"Treino -> AUC: {auc_train:.4f} | GINI: {gini_train:.4f} | KS: {ks_train:.4f}")
print(f"OOS    -> AUC: {auc_oos:.4f} | GINI: {gini_oos:.4f} | KS: {ks_oos:.4f}")
print("="*30)

# Verificação de Overfitting
if (gini_train - gini_oos) > 0.05:
    print("⚠️ ALERTA: Gap de Gini alto. Pode haver overfitting.")
else:
    print("✅ Modelo com generalização saudável.")


In [ ]:
# ==================================================
# 7. ANÁLISE DE VARIÁVEIS (Feature Selection)
# ==================================================
# 1. FEATURE IMPORTANCE (CORRIGIDO)
# model.feature_name_ contém exatamente as features que o modelo usou
importance_df = pd.DataFrame({
    'feature': model.feature_name_,  # <--- Use isso para garantir o tamanho igual
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

# Salvar CSV
import os
os.makedirs("v_06_final/metrics", exist_ok=True)
importance_df.to_csv("v_06_final/metrics/feature_importance.csv", index=False)

print("Top 10 Variáveis (Gain):")
print(importance_df.head(10))

In [ ]:
# ==================================================
# FEATURE SELECTION (CORRIGIDO E SEGURO)
# ==================================================
import numpy as np

# 1. Garantir que X_train seja numérico (Label Encoding automático para seleção)
# Criamos uma cópia temporária SÓ para calcular correlação e importância
X_temp = X_train.copy()

for col in X_temp.columns:
    if X_temp[col].dtype == 'object' or str(X_temp[col].dtype) == 'category':
        # Converte categóricas/strings para códigos inteiros
        X_temp[col] = X_temp[col].astype('category').cat.codes
    elif X_temp[col].dtype == 'float64':
        X_temp[col] = X_temp[col].astype('float32')

# 2. Filtrar importância > 0
# 'importance_df' é a tabela que você gerou após o treino
features_imp = importance_df[importance_df['importance'] > 0]['feature'].tolist()
print(f"Variáveis com importância > 0: {len(features_imp)}")

# 3. Definir Features "Protegidas" (Scores não podem ser removidos pela correlação)
features_protegidas = ['SCORE_01', 'SCORE_02']

# 4. Selecionar Top N (Garantindo que existem no dataframe)
features_imp_sorted = importance_df[importance_df['feature'].isin(X_temp.columns)].sort_values(by='importance', ascending=False)['feature'].tolist()
# Adicionei .isin(X_temp.columns) para evitar erro se alguma feature do importance_df tiver sido dropada do X_temp

# Pegamos as top 300 + as protegidas
features_top = features_imp_sorted[:300]
for p in features_protegidas:
    if p not in features_top and p in X_temp.columns: # Verifica se existe
        features_top.append(p)

# Filtrar X_temp apenas com essas features
X_top = X_temp[features_top]

# 5. Calcular Correlação e Remover Redundantes
print("Calculando matriz de correlação...")
corr_matrix = X_top.corr().abs()

# Triângulo superior
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Identificar features para dropar (Corr > 0.90)
# Mas PROTEGEMOS os Scores
to_drop = []
for column in upper.columns:
    # Se correlação alta
    if any(upper[column] > 0.90):
        # Se não for protegida, marca para dropar
        if column not in features_protegidas:
            to_drop.append(column)

print(f"Variáveis redundantes removidas: {len(to_drop)}")

# 6. Lista Final de Features Selecionadas
features_selected = [f for f in features_top if f not in to_drop]

print(f"\nTotal Final de Features: {len(features_selected)}")
print(f"Score 01 na lista: {'SCORE_01' in features_selected}")
print(f"Score 02 na lista: {'SCORE_02' in features_selected}")

In [ ]:
# 1. Criar novos datasets
X_train_sel = X_train[features_selected].copy()
X_oos_sel = X_oos[features_selected].copy()

# 2. (PRIORITÁRIO) Converter SCORES para Float32 ANTES de tratar o resto como categoria
# Se deixarmos para depois, eles podem virar 'categoria' no loop abaixo.
for col in ['SCORE_01', 'SCORE_02']:
    if col in X_train_sel.columns:
        X_train_sel[col] = pd.to_numeric(X_train_sel[col], errors='coerce').astype('float32')
        X_oos_sel[col] = pd.to_numeric(X_oos_sel[col], errors='coerce').astype('float32')

# 3. Converter o RESTANTE para 'category' do pandas
for col in X_train_sel.columns:
    # Pula se já for float32 (para não reprocessar os Scores que acabamos de arrumar)
    if X_train_sel[col].dtype == 'float32':
        continue

    if X_train_sel[col].dtype == 'object' or X_train_sel[col].dtype.name == 'category':
        X_train_sel[col] = X_train_sel[col].astype('category')
        X_oos_sel[col] = X_oos_sel[col].astype('category')

# 4. Reduzir memória (Float32) - Para o restante dos numéricos que sobraram
for col in X_train_sel.select_dtypes(include=['float64']).columns:
    X_train_sel[col] = X_train_sel[col].astype('float32')
    X_oos_sel[col] = X_oos_sel[col].astype('float32')

In [ ]:
print(f"Shape Treino Refinado: {X_train_sel.shape}")

In [ ]:
# 1. Filtra a importância para considerar apenas o que já foi selecionado
imp_final = importance_df[importance_df['feature'].isin(features_selected)]

# 2. Seleciona as Top 50
features_top_50 = imp_final.sort_values(by='importance', ascending=False)['feature'].head(50).tolist()

# 3. (Opcional mas recomendado) Garante que os Scores não saíram da lista
for score in ['SCORE_01', 'SCORE_02']:
    if score in features_selected and score not in features_top_50:
        features_top_50.append(score) # Adiciona o score se ele não estiver no Top 50
        # Se quiser manter estritamente 50, remova o último item: features_top_50.pop()

print(f"Novo total de features para treino: {len(features_top_50)}")

In [ ]:
# 1. Criar novos datasets
X_train_sel = X_train[features_top_50].copy()
X_oos_sel = X_oos[features_top_50].copy()

# 2. (PRIORITÁRIO) Converter SCORES para Float32 ANTES de tratar o resto como categoria
# Se deixarmos para depois, eles podem virar 'categoria' no loop abaixo.
for col in ['SCORE_01', 'SCORE_02']:
    if col in X_train_sel.columns:
        X_train_sel[col] = pd.to_numeric(X_train_sel[col], errors='coerce').astype('float32')
        X_oos_sel[col] = pd.to_numeric(X_oos_sel[col], errors='coerce').astype('float32')

# 3. Converter o RESTANTE para 'category' do pandas
for col in X_train_sel.columns:
    # Pula se já for float32 (para não reprocessar os Scores que acabamos de arrumar)
    if X_train_sel[col].dtype == 'float32':
        continue

    if X_train_sel[col].dtype == 'object' or X_train_sel[col].dtype.name == 'category':
        X_train_sel[col] = X_train_sel[col].astype('category')
        X_oos_sel[col] = X_oos_sel[col].astype('category')

# 4. Reduzir memória (Float32) - Para o restante dos numéricos que sobraram
for col in X_train_sel.select_dtypes(include=['float64']).columns:
    X_train_sel[col] = X_train_sel[col].astype('float32')
    X_oos_sel[col] = X_oos_sel[col].astype('float32')

In [ ]:
print(f"Shape Treino Refinado: {X_train_sel.shape}")

In [ ]:
# ==================================================
# 6. TREINO COM FEATURES SELECIONADAS (REFINADO)
# ==================================================

# 5. Treinar
# Definir a lista de categóricas baseada no que REALMENTE existe no X_train_sel agora
cat_cols_refined = list(X_train_sel.select_dtypes(include=['category']).columns)

# 4. Treinar com a MESMA configuração
model_refined = lgb.LGBMClassifier(
    random_state=SEED,
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=5,
    min_child_samples=100,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.5,
    reg_lambda=3.0,
    objective='binary',
    metric='auc',
    n_jobs=4,
    verbose=-1,
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
)

model_refined.fit(
    X_train_sel, y_train,
    eval_set=[(X_oos_sel, y_oos)],
    eval_metric='auc',
    categorical_feature=cat_cols_refined, # <--- Use a lista dinâmica, não a antiga 'all_cat_features'
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
)

print("Treinamento do modelo refinado concluído.")

In [ ]:
# ==================================================
# 6. MÉTRICAS E AVALIAÇÃO
# ==================================================
def calculate_ks(y_true, y_proba):
    """Calcula o Kolmogorov-Smirnov"""
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    ks = max(tpr - fpr)
    return ks

# Previsões
y_pred_train = model_refined.predict_proba(X_train_sel)[:, 1]
y_pred_oos = model_refined.predict_proba(X_oos_sel)[:, 1]

# Cálculos
auc_train = roc_auc_score(y_train, y_pred_train)
auc_oos = roc_auc_score(y_oos, y_pred_oos)

ks_train = calculate_ks(y_train, y_pred_train)
ks_oos = calculate_ks(y_oos, y_pred_oos)

gini_train = 2 * auc_train - 1
gini_oos = 2 * auc_oos - 1

# Impressão dos Resultados
print("\n" + "="*30)
print("   RESULTADOS DO MODELO V_06")
print("="*30)
print(f"Treino -> AUC: {auc_train:.4f} | GINI: {gini_train:.4f} | KS: {ks_train:.4f}")
print(f"OOS    -> AUC: {auc_oos:.4f} | GINI: {gini_oos:.4f} | KS: {ks_oos:.4f}")
print("="*30)

# Verificação de Overfitting
if (gini_train - gini_oos) > 0.05:
    print("⚠️ ALERTA: Gap de Gini alto. Pode haver overfitting.")
else:
    print("✅ Modelo com generalização saudável.")


In [ ]:
# ==================================================
# 8. SALVAR ARTEFATOS NO OCI
# ==================================================
print("\nSalvando artefatos no OCI...")

# Criar estrutura de pastas local temporária
os.makedirs("v_06_final/artifacts", exist_ok=True)
os.makedirs("v_06_final/metrics", exist_ok=True)
os.makedirs("v_06_final/dataset", exist_ok=True)

# 1. Salvar Modelo (Pickle)
joblib.dump(model_refined, "v_06_final/artifacts/model_lgbm.pkl")

# 2. Salvar Métricas (JSON)
metrics_dict = {
    "modelo": "v_06_final",  # Atualize o nome
    "features": features_top_50, # Atualize a lista
    "auc_train": round(auc_train, 4),
    "gini_train": round(gini_train, 4),
    "ks_train": round(ks_train, 4),
    "auc_oos": round(auc_oos, 4),
    "gini_oos": round(gini_oos, 4),
    "ks_oos": round(ks_oos, 4),
    "best_iteration": model_refined.best_iteration_,
    "seed": SEED
}

with open("v_06_final/metrics/metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=4)

# 3. Upload para o OCI
fs = ocifs.OCIFileSystem()

# Upload Modelo
fs.put("v_06_final/artifacts/model_lgbm.pkl", PATH_MODELS + "artifacts/model_lgbm.pkl")
print(f"Modelo salvo em: {PATH_MODELS}artifacts/model_lgbm.pkl")

# Upload Métricas
fs.put("v_06_final/metrics/metrics.json", PATH_MODELS + "metrics/metrics.json")
print(f"Métricas salvas em: {PATH_MODELS}metrics/metrics.json")

print("\n✅ Processo finalizado com sucesso!")

In [ ]:
# ==================================================
# 9. SALVAR DATASETS (Reprodutibilidade)
# ==================================================
print("Salvando datasets de treino e validação...")

# Salva em Parquet (formato eficiente)
X_train_sel.to_parquet("v_06_final/dataset/train.parquet", index=False)
X_oos_sel.to_parquet("v_06_final/dataset/oos.parquet", index=False)

# Upload para o OCI
fs.put("v_06_final/dataset/train.parquet", PATH_MODELS + "dataset/train.parquet")
fs.put("v_06_final/dataset/oos.parquet", PATH_MODELS + "dataset/oos.parquet")

print("Datasets salvos com sucesso.")

In [ ]:
# Limpa todos os outputs do notebook atual
!jupyter nbconvert nome_do_seu_notebook.ipynb --to notebook --clear-output --output modelo_final_limpo.ipynb